# Return (trend) + risk (EWMA σ) with real data from the repo

**Headline:** *"the model predicts a return of X %"* at H = 21 days, evaluated with **MAE, RMSE and R²** against two baselines (predicting 0 % and the historical mean).
Only **two numbers per asset** are kept from the model: `mu` (expected return) and `sigma` (expected volatility).
With them, `decision/` and the simulation use a **normal approximation**.

- **Return:** **one-factor trend** model (*time-series momentum*): average momentum over 1, 3, 6 and 12 months → a line with 2 parameters.
- **Risk:** **EWMA σ with λ = 0.97**, validated with the coverage of the 90 % interval.
- Ridge (7 features) is kept only for comparison.

Everything with **NumPy + pandas** (rule in `model/README.md`: no ML libraries).

Put it in the repo's `notebooks/` folder and run it top to bottom.

## 1. Setup and access to real data

In [ ]:
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from math import erf, sqrt
warnings.filterwarnings("ignore")

SOURCE = "real"
H          = 21        # horizon in days (bars of each asset): at 5 days there is no signal, at 21 there is
LAMBDA     = 0.97      # EWMA decay (better calibrated than 0.94 at 5, 21 and 63 days)
ALPHA      = 20000.0   # Ridge regularisation (comparison only)
TEST_FROM  = "2000-01-01"   # before 2000 there is almost only ^GSPC, AAPL and MSFT
SEED       = 42
rng = np.random.default_rng(SEED)

# Only access to the repo: data/fetch.py (cached prices). Searched upwards from this folder.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "fetch.py").exists()), None)
if ROOT is not None:
    sys.path.insert(0, str(ROOT))
try:
    from data.assets import all_symbols, label
    from data.fetch import get_close
except ImportError:
    all_symbols = get_close = None
    label = lambda s: s
    if SOURCE == "real":
        raise ImportError("For SOURCE = 'real' open this notebook inside the OxHack2026 repo (it needs data/fetch.py).")
from data.fetch import fetch_many, load   # same module; fetch_many downloads in batch (Yahoo blocks one-by-one requests)
print("Repo:", ROOT)

In [ ]:
SYMBOLS = all_symbols()

# 1) If any is missing from the cache, download ALL missing ones in a single batch.
missing = [s for s in SYMBOLS if load(s) is None]
if missing:
    print("Downloading (once):", missing)
    fetch_many(missing)

# 2) get_close(symbol) -> pd.Series of daily closes (reads from the cache)
closes = {}
for s in SYMBOLS:
    try:
        c = get_close(s).dropna()
        c = c[c > 0]
        if len(c) > 500:
            closes[s] = c
        else:
            print("  skipped (short history):", s)
    except Exception as e:
        print("  skipped", s, "->", type(e).__name__, e)

print(f"{len(closes)} assets with data")
pd.DataFrame({s: dict(start=c.index.min().date(), end=c.index.max().date(), rows=len(c), last=round(c.iloc[-1], 2))
              for s, c in closes.items()}).T

## 2. Returns, target and EWMA volatility
Each asset is handled on **its own calendar** (crypto 365 days, stock markets 252). The target is the
cumulative log return over the next H bars: `y_t = log(P_{t+H}/P_t)`.

To mix assets that are so different (BTC vs. an exchange rate), **everything is scaled by EWMA volatility**:
`z = y / (σ_daily·√H)`. The model predicts `z`, and then `mu = z·σ·√H`.

Features: `TREND` (momentum at 21, 63, 126 and 252 days) feeds the trend model; `FEATS` (7 features) feeds Ridge.

In [ ]:
def ewma_vol(r, lam=LAMBDA):
    """Daily σ: σ²_t = λ·σ²_{t-1} + (1-λ)·r²_t, using data up to t."""
    return np.sqrt(r.pow(2).ewm(alpha=1-lam, adjust=False, min_periods=30).mean())

def build(s, c):
    r = np.log(c).diff().dropna()
    v = ewma_vol(r)                                   # daily σ up to t
    sH = v * np.sqrt(H)                               # σ over H days
    f = pd.DataFrame(index=r.index)
    f["lag1"]   = r / v
    for k in (5, 21, 63, 126, 252):
        f[f"mom{k}"] = r.rolling(k).sum() / (v*np.sqrt(k))
    f["lvol"]   = np.log(v / r.rolling(63).std())     # is volatility rising or falling?
    cum = r.cumsum(); f["dd63"] = (cum - cum.rolling(63).max()) / (v*np.sqrt(63))
    f["y"]      = np.log(c.shift(-H) / c).reindex(r.index)
    f["sigma"]  = sH
    f["asset"]  = s
    return f

panel = pd.concat([build(s, c) for s, c in closes.items()])
panel.index.name = "date"
FEATS = ["lag1","mom5","mom21","mom63","mom126","lvol","dd63"]   # Ridge (comparison)
TREND = ["mom21","mom63","mom126","mom252"]                      # trend model
panel[FEATS + ["mom252"]] = panel[FEATS + ["mom252"]].clip(-5, 5)   # clip extremes
dataset = panel.dropna().reset_index().sort_values(["date","asset"]).reset_index(drop=True)
dataset["z"] = dataset["y"] / dataset["sigma"]
print(dataset.shape, dataset.date.min().date(), "→", dataset.date.max().date())
dataset.groupby("asset").size().rename("obs").to_frame().T

## 3. Time split (walk-forward with *purge*)
Dates are never shuffled. The test starts at `TEST_FROM` (2000) and is split into 6 blocks; each block is
predicted with a model trained only on the past. A `2H`-day gap is left between training and test so that
no training label (which looks H bars into the future) overlaps the test.

In [ ]:
dates = np.sort(dataset.date.unique())
N_SPLITS, GAP = 6, pd.Timedelta(days=2*H+2)

def walk_forward():
    n = len(dates); start = np.searchsorted(dates, np.datetime64(TEST_FROM)); step = (n-start)//N_SPLITS
    for k in range(N_SPLITS):
        a = start + k*step; b = a + step if k < N_SPLITS-1 else n
        t0 = dates[a]; t1 = dates[b-1]
        yield (dataset.date < t0 - GAP), (dataset.date >= t0) & (dataset.date <= t1)

for i,(mtr,mte) in enumerate(walk_forward()):
    print(f"fold {i}: train until {dataset.date[mtr].max().date()} ({mtr.sum():,}) | "
          f"test {dataset.date[mte].min().date()}→{dataset.date[mte].max().date()} ({mte.sum():,})")

## 4. Models (pure NumPy)
- **Zero**: predicts 0 %.
- **Historical mean**: mean of `z` in training (captures only the drift: "markets go up on average").
- **Ridge**: 7 features, `β = (XᵀX + αI)⁻¹Xᵀz`. Kept for comparison.
- **Trend (chosen model)**: a single signal `s = mean(mom21, mom63, mom126, mom252)` and a line
  `z = a + b·s` fitted by least squares. It concentrates *time-series momentum* in one factor (assets that have been
  rising tend to keep rising for weeks or months; Moskowitz, Ooi and Pedersen, 2012).

In [ ]:
def ridge_fit(X, z, alpha=ALPHA):
    mu, sd = X.mean(0), X.std(0) + 1e-9
    Xs = (X - mu) / sd
    zc = z.mean()
    beta = np.linalg.solve(Xs.T @ Xs + alpha*np.eye(X.shape[1]), Xs.T @ (z - zc))
    return dict(mu=mu, sd=sd, beta=beta, intercept=zc)

def ridge_predict(m, X):
    return m["intercept"] + ((X - m["mu"]) / m["sd"]) @ m["beta"]

def trend_fit(X, z):
    s = X.mean(1)
    b = np.cov(s, z)[0, 1] / s.var()
    return dict(a=z.mean() - b*s.mean(), b=b)

def trend_predict(m, X):
    return m["a"] + m["b"] * X.mean(1)

MODELS = ["Zero", "Historical mean", "Ridge", "Trend"]
zhat = {k: np.full(len(dataset), np.nan) for k in MODELS}
test_mask = np.zeros(len(dataset), bool)
for mtr, mte in walk_forward():
    tr, te = dataset[mtr], dataset[mte]
    zhat["Zero"][mte]            = 0.0
    zhat["Historical mean"][mte] = tr["z"].mean()
    zhat["Ridge"][mte]           = ridge_predict(ridge_fit(tr[FEATS].values, tr["z"].values), te[FEATS].values)
    zhat["Trend"][mte]           = trend_predict(trend_fit(tr[TREND].values, tr["z"].values), te[TREND].values)
    test_mask |= mte.values

oos = dataset[test_mask].copy()
for k, v in zhat.items():
    oos[k] = v[test_mask] * oos["sigma"]          # back to return units: mu = ẑ·σ√H
print("Out-of-sample observations:", len(oos))

## 5. Evaluation: MAE, RMSE and R²
Reported in percentage points (pp). **An R² of 0–2 % for returns is normal**; what matters is beating the baselines.
- `R² vs zero`: improvement over always predicting 0 %.
- `R² vs mean`: improvement over the historical mean. **This is the demanding test**: it separates the model's signal from the plain upward drift.
- `% years beating mean`: stability; in how many years the model had a lower error than the mean.

Sign accuracy is not reported: the historical mean, which always predicts "up", is already right ~55 % of the time just from the drift.

In [ ]:
def metrics(y, p, base):
    e = y - p
    return dict(MAE=np.mean(np.abs(e))*100, RMSE=np.sqrt(np.mean(e**2))*100,
                R2=1 - np.sum(e**2)/np.sum((y - y.mean())**2),
                R2_vs_zero=1 - np.sum(e**2)/np.sum(y**2),
                R2_vs_mean=1 - np.sum(e**2)/np.sum((y - base)**2))

def years_beating(g, k):
    a = g.groupby(g.date.dt.year).apply(lambda x: ((x.y - x[k])**2).sum() < ((x.y - x["Historical mean"])**2).sum())
    return a.mean()

y, base = oos["y"].values, oos["Historical mean"].values
table = pd.DataFrame({k: {**metrics(y, oos[k].values, base), "years": years_beating(oos, k)} for k in MODELS}).T
table.columns = ["MAE (pp)","RMSE (pp)","R²","R² vs zero","R² vs mean","% years beating mean"]
table.round(4)

In [ ]:
# By asset (Trend vs. Historical mean): where is the signal?
rows = {}
for a, g in oos.groupby("asset"):
    mt = metrics(g.y.values, g.Trend.values, g["Historical mean"].values)
    mm = metrics(g.y.values, g["Historical mean"].values, g["Historical mean"].values)
    rows[label(a)] = dict(RMSE_pp=mt["RMSE"], RMSE_mean_pp=mm["RMSE"],
                          R2_vs_mean=mt["R2_vs_mean"], R2_vs_zero=mt["R2_vs_zero"])
pd.DataFrame(rows).T.sort_values("R2_vs_mean", ascending=False).round(4)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11,4))
ax[0].scatter(oos.Trend*100, oos.y*100, s=2, alpha=.2)
lim = np.percentile(np.abs(oos.y*100), 99); ax[0].plot([-lim,lim],[-lim,lim],"r--",lw=1)
ax[0].set(xlim=(-lim,lim), ylim=(-lim,lim), xlabel="Predicted (%)", ylabel="Actual (%)", title=f"Trend: predicted vs actual at {H} days")
m_ = oos.assign(err_t=(oos.y-oos.Trend)**2, err_m=(oos.y-oos["Historical mean"])**2).set_index("date").resample("YS")[["err_t","err_m"]].mean()
(np.sqrt(m_)*100).plot(ax=ax[1]); ax[1].set(title="RMSE by year (pp)", xlabel=""); ax[1].legend(["Trend","Historical mean"])
plt.tight_layout(); plt.show()

### 5b. Alternative models tested (and why they are not used)
With the same walk-forward (test from 2000, λ = 0.97), more complex models were tested with scikit-learn, outside this
notebook because the team rule is not to use ML libraries. Out-of-sample `R² vs mean`:

| Model | 5 days | 21 days | 63 days |
|---|---|---|---|
| Ridge α=20000 (7 features) | +0.06 % | +0.09 % | +1.01 % |
| Gradient Boosting, default settings | −3.3 % | −5.0 % | −6.3 % |
| Gradient Boosting, heavily regularised | +0.17 % | +0.20 % | +0.70 % |
| Random Forest, regularised | +0.17 % | +0.18 % | +0.47 % |
| Ridge + GBR + RF ensemble | +0.16 % | +0.22 % | +0.81 % |
| Any model with 19 features (market momentum, relative momentum, MA200, asset class) | worse | worse | much worse |
| **One-factor trend** | +0.16 % | **+0.52 %** | **+1.04 %** |

- Unregularised trees **overfit**: they learn noise and lose to the mean almost every year.
- Regularised, they barely match a linear model: the signal available in prices is **linear and weak** (momentum).
- More features and more flexibility always make things worse. The simplest model wins because it concentrates the only real signal in one factor.
- To truly improve, **new data** is needed (rates, valuation, macro), not a more complex algorithm.

## 6. Risk: a single number, volatility (EWMA σ)
`σ_H = σ_daily_EWMA · √H`. Validated with **calibration**: with `z = (y − mu)/σ`,
the standard deviation of `z` should be ≈ 1 and the 90 % interval should cover ≈ 90 % of cases.
Compared against **simple historical volatility** (README baseline) and several λ. The model uses λ = 0.97.

In [ ]:
def coverage(y, mu, sig, level=0.90):
    from scipy.stats import norm
    z = (y - mu)/sig; zc = norm.ppf(0.5 + level/2)
    return dict(std_z=z.std(), coverage_90=(np.abs(z) <= zc).mean(),
                logLikelihood=np.mean(-np.log(sig) - 0.5*z**2))

candidates = {}
for lam in (0.94, 0.97, 0.99):
    s = pd.concat([ewma_vol(np.log(c).diff().dropna(), lam).rename(a) * np.sqrt(H) for a, c in closes.items()],
                  keys=closes.keys())
    candidates[f"EWMA λ={lam}"] = s
hist = pd.concat([np.log(c).diff().dropna().rolling(63).std().rename(a) * np.sqrt(H) for a, c in closes.items()], keys=closes.keys())
candidates["Simple hist. 63d"] = hist

idx = pd.MultiIndex.from_frame(oos[["asset","date"]])
calib = {}
for name, s in candidates.items():
    sg = s.reindex(idx).values
    ok = np.isfinite(sg) & (sg > 0)
    calib[name] = coverage(oos.y.values[ok], oos.Trend.values[ok], sg[ok])
calib = pd.DataFrame(calib).T
calib.round(3)

In [ ]:
a = "SPY" if "SPY" in closes else list(closes)[0]
g = dataset[dataset.asset == a].set_index("date")
fig, ax = plt.subplots(figsize=(10,3))
(g.sigma*100).plot(ax=ax, label=f"EWMA σ over {H} days")
(g.y.abs()*100).rolling(21).mean().plot(ax=ax, alpha=.6, label="mean |actual return| 21d")
ax.set(title=f"Estimated risk — {label(a)}", ylabel="%", xlabel=""); ax.legend(); plt.show()

## 7. Final model and `predict` function with the team contract
The **trend** model is retrained on the **whole** history and `predict(prices, horizon_days)` is defined with the signature in `model/README.md`:
`rendimiento_esperado`, `volatilidad`, `prob_perdida`, `intervalo` (90 %).
For a horizon other than H, `mu` scales linearly and `σ` with √horizon.

In [ ]:
from math import erf, sqrt
Phi = lambda x: 0.5*(1 + erf(x/sqrt(2)))
Z90 = 1.6448536269514722

MODEL = trend_fit(dataset[TREND].values, dataset["z"].values)
print(f"Trend model: z = {MODEL['a']:.4f} + {MODEL['b']:.4f} · mean({', '.join(TREND)})")

def predict(prices: pd.Series, horizon_days: int = H) -> dict:
    """model/ contract: prices = daily closes with the date as index."""
    c = prices.dropna(); c = c[c > 0]
    f = build("x", c).iloc[-1]                         # last row: only information up to today
    x = np.clip(f[TREND].astype(float).values, -5, 5)
    z = float(trend_predict(MODEL, x[None, :])[0])
    sig_d = float(ewma_vol(np.log(c).diff().dropna()).iloc[-1])
    sigma = sig_d * sqrt(horizon_days)
    mu = z * sig_d * sqrt(H) * (horizon_days / H)
    return {"rendimiento_esperado": mu, "volatilidad": sigma,
            "prob_perdida": Phi(-mu / sigma),
            "intervalo": (mu - Z90*sigma, mu + Z90*sigma)}

## 8. Current forecast: "the model predicts a return of X %"

In [ ]:
rows = {}
for s, c in closes.items():
    p = predict(c, H)
    rows[s] = dict(asset=label(s), last_date=c.index[-1].date(), mu=p["rendimiento_esperado"],
                   sigma=p["volatilidad"], p_loss=p["prob_perdida"],
                   ci90_low=p["intervalo"][0], ci90_high=p["intervalo"][1])
forecast = pd.DataFrame(rows).T
for col in ["mu","sigma","p_loss","ci90_low","ci90_high"]:
    forecast[col] = forecast[col].astype(float)
for s, r in forecast.iterrows():
    print(f"{r.asset:<34} the model predicts a return of {r.mu*100:+.2f} %  (σ {r.sigma*100:.2f} %, P(loss) {r.p_loss:.0%}, "
          f"CI90 [{r.ci90_low*100:+.1f} %, {r.ci90_high*100:+.1f} %])")

## 9. Monte Carlo simulation (normal with correlations)
`R ~ N(mu, Σ)` with `Σ = D·ρ·D`, `D = diag(σ)` and `ρ` the correlation over the last year
(pairwise, on the days when both assets trade).

In [ ]:
N_SIM = 20000
rets = pd.DataFrame({s: np.log(c).diff() for s, c in closes.items()}).tail(365)
corr = np.array(rets.corr(min_periods=60).fillna(0.0).values, dtype=float)
np.fill_diagonal(corr, 1.0)
# project onto a positive semi-definite matrix (pairwise correlations may not be)
w, V = np.linalg.eigh(corr); corr = (V * np.clip(w, 1e-6, None)) @ V.T
d = np.sqrt(np.diag(corr)); corr = corr / np.outer(d, d)

sig = forecast.sigma.values; mu = forecast.mu.values
sim = pd.DataFrame(rng.multivariate_normal(mu, np.diag(sig) @ corr @ np.diag(sig), size=N_SIM), columns=forecast.index)

portfolio = sim.mean(axis=1)                     # example: equal weight
print(f"Equal-weight portfolio over {H} days: mean {portfolio.mean()*100:+.2f} % | "
      f"VaR95 {-np.percentile(portfolio,5)*100:.2f} % | P(loss) {(portfolio<0).mean():.0%}")
fig, ax = plt.subplots(figsize=(7,3)); ax.hist(portfolio*100, bins=80, alpha=.8); ax.axvline(0, color="k", lw=1)
ax.set(title="Simulated portfolio distribution", xlabel="Return (%)"); plt.show()

## 10. Export
Two numbers per asset (`mu`, `sigma`) + horizon; everything else is derived with the normal.

In [ ]:
out = (ROOT / "notebooks" / "outputs") if ROOT else Path("outputs"); out.mkdir(parents=True, exist_ok=True)
forecast[["mu","sigma"]].assign(horizon_days=H).to_csv(out / "pronostico_mu_sigma.csv")
table.to_csv(out / "metricas_oos.csv"); calib.to_csv(out / "calibracion_sigma.csv")
print("Saved to", out)

### Notes
- **Mixed calendars:** crypto trades 7 days a week and stock markets 5, so `H` counts bars of each asset (≈ 1 month for stocks, 3 weeks for crypto). Correlation is computed pairwise.
- **London** is quoted in pence; this does not affect relative returns.
- **Fat tails:** if `coverage_90` falls below 0.90, the normal underestimates extreme risk; σ can be widened with a factor.
- **The signal is weak but real.** An `R² vs mean` of ~0.5 % at 21 days is normal for returns; the interval will still be much wider than `mu`. The defensible message: *"predicts X % over one month, better than the historical mean out of sample, with a risk interval calibrated at 90 %"*.
- **Why H = 21:** at 5 days no model beats the mean; at 63 days the signal is larger but there are few independent windows since 2000.